In [1]:
import json, os, asyncio, pdb
import pandas as pd
from pathlib import Path
import pyprojroot, sys
from pyprojroot.criterion import has_file
root = pyprojroot.find_root(has_file("pyproject.toml"))

sys.path.insert(0, str(root))
os.chdir(root)  #
from dotenv import load_dotenv

In [2]:
load_dotenv()
from kdutils.macro import base_path


In [3]:
trade_date = "2026-10-16"

In [4]:
filename = os.path.join(base_path,"live", "temp", trade_date, "predict",
                        "{0}_000852.json".format(trade_date))

In [5]:
with open(filename, "r", encoding="utf-8") as f:
    snapshot = json.load(f)

In [6]:
snapshot['trader_prediction']

{'dominant_regime': '外部系统性冲击引发恐慌急跌后的高波动修复期：宽度分位自 0.004 历史极值单日修复至 0.63，估值分位 0.0027 与仓位箱体 0.04 处 252 日极低水位，国内政策底与全球科技股杀估值多空对峙、高赔率均值回归背景',
 'predict_direction': 'UP',
 'confidence': 0.6,
 'used_features': ['main_flow_ratio',
  'basis_diff_3d',
  'basis_div_strength',
  'panic_reversal_score',
  'exhaustion_intensity',
  'TF-20261016-002',
  'TF-20261015-001',
  'TF-20261016-004'],
 'predict_logic': [{'feature': 'main_flow_ratio',
   'feature_layer': 'PREDICT',
   'observation': '主力资金比在 10-14/10-15 连续深负（-0.29/-0.41）后，10-16 骤转强正 +0.3787，为四日窗口内首次净流入',
   'inference': "恐慌抛售峰值后主动买盘回归，资金面出现方向性拐点，为超跌反弹提供增量燃料；同时满足历史案例中'重新做多'的两项前置确认条件（主力转正+宽度修复），保守 FLAT 约束解除"},
  {'feature': 'basis_div_strength',
   'feature_layer': 'PREDICT',
   'observation': '基差 3 日差分连续 4 日为正（0.31/0.32/0.30/0.33），基差背离强度持续维持 0.42-0.43 高位，基差分位处于 252 日 100% 极值',
   'inference': "期货端杠杆资金在恐慌极值区持续领先现货定价修复，衍生品多头结构构成'市场结构确认（EV2类）'，与政策底信号形成同日共振的验证链"},
  {'feature': 'panic_reversal_score',
   'feature_layer': 'PREDICT',
   'observation':

In [7]:
from typing import Dict, Any, List

FEATURE_MAP = {
    "main_flow_acceleration": "主力资金加速度（二阶动量）",
    "main_flow_ratio": "主力资金净流入比率",
    "price_acc_5d": "5日价格动量与加速度",
    "breadth_div_intensity_pos": "市场宽度与正向背离",
    "oi_change_3d": "衍生品持仓量变动（3D OI）",
    "panic_reversal_score": "恐慌反转得分",
    "basis_div_strength": "期货基差背离强度",
    "global_liquidity": "全球流动性宏观环境",
    "domestic_policy": "国内政策对冲与托底",
}


class TradingMemoGenerator:
    """策略研判与交易决策备忘录生成器（执行与风控先行版）"""

    def __init__(self, data: Dict[str, Any]):
        self.data = data
        self.direction = data.get("predict_direction", "NEUTRAL")
        self.confidence = data.get("confidence", 0.0)

    def _render_header(self) -> str:
        dir_badge = {
            "DOWN": "偏空（DOWN）🔻",
            "UP": "偏多（UP）🔺",
            "NEUTRAL": "中性观望（NEUTRAL）⚪",
        }.get(self.direction, self.direction)

        conf_tag = (
            "高置信度"
            if self.confidence >= 0.75
            else "中等置信度"
            if self.confidence >= 0.55
            else "低置信度"
        )

        return f"""# 策略研判与交易决策备忘录

* **交易基准日**：`{self.data.get('trade_time', 'N/A')}`
* **研判模块**：`{self.data.get('name', 'N/A')}`
* **核心方向**：**{dir_badge}**
* **置信度**：`{self.confidence:.2f}`（{conf_tag}）
"""

    def _render_execution_summary(self) -> str:
        """一、交易指令摘要（提前至首位）"""
        if self.direction == "DOWN":
            stance = "轻仓试空或收缩多头仓位；由于政策对冲存在，**严禁激进追空**。"
        elif self.direction == "UP":
            stance = "逢低偏多布局，警惕上方压力。"
        else:
            stance = "多空方向不明，维持区间中性交易。"

        return f"""### 一、 交易指令摘要（Execution Summary）

* **战术头寸建议**：{stance}
* **核心风控盯盘**：重点关注主力资金是否二阶回正、基差背离强弱及政策落地长阳催化。
"""

    def _render_invalid_conditions(self) -> str:
        """二、风控与逻辑失效边界（提前至状态前）"""
        conds = self.data.get("invalid_conditions", [])
        if not conds:
            return ""

        lines = [
            "### 二、 风控与逻辑失效边界（Invalid Conditions）\n",
            "交易执行必须严格遵守以下**失效平仓/止损与反手预案**：\n",
        ]
        for i, cond in enumerate(conds, start=1):
            lines.append(f"{i}. {cond}")
        lines.append("")
        return "\n".join(lines)

    def _render_regime(self) -> str:
        """三、市场主导状态"""
        regime = self.data.get("dominant_regime", "暂无定性描述")
        return f"""### 三、 市场主导状态（Dominant Regime）

> **{regime}**
"""

    def _render_predict_logic(self) -> str:
        """四、多维量化信号与逻辑推演"""
        logics: List[Dict[str, Any]] = self.data.get("predict_logic", [])
        if not logics:
            return ""

        quant_signals = [l for l in logics if l.get("feature_layer") == "PREDICT"]
        macro_signals = [l for l in logics if l.get("feature_layer") == "TEXTUAL"]

        lines = ["### 四、 多维量化信号与逻辑推演（Predict Logic）\n"]

        if quant_signals:
            lines.append("#### 1. 量化微观结构与衍生品特征")
            for item in quant_signals:
                feat = item.get("feature", "")
                title = FEATURE_MAP.get(feat, feat)
                obs = item.get("observation", "")
                inf = item.get("inference", "")
                lines.append(f"- **{title}**：{obs}\n  - *推演逻辑*：{inf}")
            lines.append("")

        if macro_signals:
            lines.append("#### 2. 宏观流动性与政策文本催化（Textual Factors）")
            for item in macro_signals:
                feat = item.get("feature", "")
                title = FEATURE_MAP.get(feat, feat)
                obs = item.get("observation", "")
                inf = item.get("inference", "")
                lines.append(f"- **{title}**：{obs}\n  - *推演逻辑*：{inf}")
            lines.append("")

        return "\n".join(lines)

    def generate(self) -> str:
        """组合生成完整 Markdown（新顺序：指令摘要 -> 风控边界 -> 主导状态 -> 信号推演）"""
        sections = [
            self._render_header(),
            self._render_execution_summary(),   # 1. 交易指令摘要
            self._render_invalid_conditions(),  # 2. 风控与失效边界
            self._render_regime(),              # 3. 市场主导状态
            self._render_predict_logic(),       # 4. 信号逻辑推演
        ]
        return "\n".join(s for s in sections if s)

In [8]:
generator = TradingMemoGenerator(snapshot['trader_prediction'])
markdown_output = generator.generate()
from rich.console import Console
from rich.markdown import Markdown

In [9]:
from rich.console import Console
from rich.markdown import Markdown
from rich.theme import Theme

# 1. 定义自定义主题，将标题和引用块强制改为白色
custom_theme = Theme({
    "markdown.h1": "bold white",           # 一级标题
    "markdown.h2": "bold white",           # 二级标题
    "markdown.h3": "bold white",           # 三级标题（如：一、市场主导状态）
    "markdown.h4": "bold white",           # 四级标题
    "markdown.block_quote": "bright_white",# 引用块竖线与内容（即图中的 ▌ 区域）
    "markdown.paragraph": "white",         # 正文段落
    "markdown.text": "white",              # 普通文本
    "markdown.item": "white",              # 列表项
})

# 2. 传入 theme 初始化 Console
console = Console(theme=custom_theme)

# 3. 正常输出即可变为纯白 / 亮白色
console.print(Markdown(markdown_output))

策略研判与交易决策备忘录                                              

 • 交易基准日：2026-10-16                                                                                          
 • 研判模块：reflection1                                                                                           
 • 核心方向：偏多（UP）🔺                                                                                          
 • 置信度：0.60（中等置信度）                                                                                      

一、 交易指令摘要（Execution Summary）                                                                             

 • 战术头寸建议：逢低偏多布局，警惕上方压力。                                                                      
 • 核心风控盯盘：重点关注主力资金是否二阶回正、基差背离强弱及政策落地长阳催化。                                    

二、 风控与逻辑失效边界（Invalid Conditions）                                                                      

交易执行必须严格遵守以下失效平仓/止损与反手预案：                                                                  

 1 基差结构回吐：T+1 起 basis_div_strength 快速跌破 0.2 或基差分位自 252 日 1.0                                    
   极值显著回落，期货端提前定价修复的核心市场结构确认失效                                                          
 2 资金拐点证伪：main_flow_ratio 重新转负且 main_flow_acceleration 持续为负（如 <                                  
   -0.3），主力回流无延续性，反弹缺燃料，退回 FLAT 观望                                                            
 3 系统性风险二次发酵：出现同级别 confirmed 的全球科技股熔断级续跌/美债收益率再创新高事件（参照                    
   TF-20261015-005、TF-20261016-003 量级），或市场宽度分位再度跌破 0.05、涨跌比跌破 0.2，恐慌二次探底，止损离场    

三、 市场主导状态（Dominant Regime）                                                                               

▌ 外部系统性冲击引发恐慌急跌后的高波动修复期：宽度分位自 0.004 历史极值单日修复至 0.63，估值分位 0.0027          
▌ 与仓位箱体 0.04 处 252 日极低水位，国内政策底与全球科技股杀估值多空对峙、高赔率均值回归背景                    

四、 多维量化信号与逻辑推演（Predict Logic）                                                                       

1. 量化微观结构与衍生品特征                                                                                        

 • 主力资金净流入比率：主力资金比在 10-14/10-15 连续深负（-0.29/-0.41）后，10-16 骤转强正                          
   +0.3787，为四日窗口内首次净流入                                                                                 
    • 推演逻辑：恐慌抛售峰值后主动买盘回归，资金面出现方向性拐点，为超跌反弹提供增量燃料；同时满足历史案例中'重新做
      多'的两项前置确认条件（主力转正+宽度修复），保守 FLAT 约束解除                                               
 • 期货基差背离强度：基差 3 日差分连续 4 日为正（0.31/0.32/0.30/0.33），基差背离强度持续维持 0.42-0.43             
   高位，基差分位处于 252 日 100% 极值                                                                             
    • 推演逻辑：期货端杠杆资金在恐慌极值区持续领先现货定价修复，衍生品多头结构构成'市场结构确认（EV2类）'，与政策底
      信号形成同日共振的验证链                                                                                     
 • 恐慌反转得分：恐慌反转得分由 10-15 的 -0.1565 转正至 10-16 的 +0.1775，exhaustion_intensity                     
   连续三日为负（-0.13~-0.20）显示抛压衰竭                                                                         
    • 推演逻辑：单边下跌动能进入衰竭段，超跌反抽的微观结构条件具备；当前处于高波修复期，反转/衰竭类信号权重高于趋势
      动量类信号                                                                                                   

2. 宏观流动性与政策文本催化（Textual Factors）                                                                     

 • 国内政策对冲与托底：TF-20261016-002                                                                             
   央行行长明确'继续适度宽松'、财政部长表态'更加积极的财政政策'，TF-20261015-001 7000亿买断式逆回购落地，此前      
   TF-20261013-001 新质生产力/AI+ 意见与 TF-20261013-002 千亿促内需专项资金构成政策催化背景                        
    • 推演逻辑：官员级宽松明示构成政策底信号，虽弱于汇金级别真金白银入场，但已与基差极值升水、主力回流、宽度修复等 
      市场结构确认同日共振，触发历史案例中'观望转短线偏多'的完整触发链                                             
 • 全球流动性宏观环境：TF-20261016-004 沃什讲话后加息概率预期回落，TF-20261016-006 IEA 释放逾 2000                 
   万桶储备压制油品冲击；但 TF-20261015-005 全球科技股大跌（KOSPI熔断）与 TF-20261016-003                          
   AI/半导体市值蒸发超万亿的系统性风险尚未完全解除                                                                 
    • 推演逻辑：外部流动性收紧与能源冲击边际缓和，为国内超跌修复打开窗口；但全球科技股杀估值风险仍在发酵，构成主要 
      下行压制，故综合置信度压至 0.60 而非案例建议区间上限                                                         